In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
portfolio = pd.read_csv("../data/portfolio.csv")
profile = pd.read_csv("../data/profile_ag.csv")
transcript = pd.read_csv("../data/transcript.csv")

---

In [3]:
display(portfolio.head(5))
display(profile.head(5))
display(transcript.head(5))

,Unnamed: 0,reward,channels,difficulty,duration,offer_type,id
0,0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


,Unnamed: 0,gender,age,id,became_member_on,income,age_group
0,0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN,100세 이상
1,1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0,50대
2,2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN,100세 이상
3,3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0,70대
4,4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN,100세 이상


,Unnamed: 0,person,event,value,time
0,0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


In [4]:
# 1. 문자열 → dict
transcript['value'] = transcript['value'].apply(ast.literal_eval)

# 2. 컬럼 확장
value_df = transcript['value'].apply(pd.Series)

# 3. 결합
transcript = pd.concat([transcript, value_df], axis=1)

# 4. offer_id 통합
transcript['offer_id'] = transcript['offer_id'].combine_first(transcript['offer id'])

# 5. 불필요 컬럼 제거
transcript = transcript.drop(columns=['value', 'offer id'])

transcript

,Unnamed: 0,person,event,time,amount,offer_id,reward
0,0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN
1,1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN
2,2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,NaN
3,3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,NaN
4,4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN
...,...,...,...,...,...,...,...
306529,306529,b3a1272bc9904337b331bf348c3e8c17,transaction,714,1.59,NaN,NaN
306530,306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,714,9.53,NaN,NaN
306531,306531,a00058cf10334a308c68e7631c529907,transaction,714,3.61,NaN,NaN
306532,306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,714,3.53,NaN,NaN


In [5]:
df = transcript.merge(
    profile,
    left_on='person',
    right_on='id',
    how='left'
)

In [7]:
df = transcript.merge(
    profile,
    left_on='person',
    right_on='id',
    how='left'
)

In [10]:
df = df.drop(columns=['Unnamed: 0_x', 'Unnamed: 0_y'])

In [11]:
df.head()

,person,event,time,amount,offer_id,reward,gender,age,id,became_member_on,income,age_group
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0,70대
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN,100세 이상
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,NaN,M,68,e2127556f4f64592b11af22de27a7932,20180426,70000.0,60대
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,118,8ec6ce2a7e7949b1bf142def7d0e0586,20170925,NaN,100세 이상
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,118,68617ca6246f4fbc85e91a2a49552598,20171002,NaN,100세 이상
